# AI Fraud & Transaction Risk Intelligence

**Portfolio-grade rare-event fraud detection that converts model scores into an investigator-ready decision system.**

### Business questions
1. Which transactions are most suspicious?
2. Which model performs best when fraud is extremely rare?
3. What operating threshold minimises an illustrative cost of false alerts vs missed fraud?
4. How much fraud can investigators capture with a fixed review capacity?
5. Why is a transaction considered high risk?

### Real dataset
This notebook uses the anonymised **Credit Card Fraud Detection** benchmark from **OpenML dataset 43627**: 284,807 European card transactions with 492 labelled frauds (~0.17%).

### Modelling stack
- Class-weighted Logistic Regression baseline
- XGBoost rare-event classifier
- Isolation Forest anomaly detection
- Hybrid supervised + anomaly risk score
- Validation-only model and threshold selection
- PR-AUC, ROC-AUC, precision, recall, F1 and capture@K
- SHAP explainability
- Ranked fraud-investigation queue

> Business-cost assumptions are illustrative and are kept separate from measured model performance.

In [ ]:
# Colab-safe setup
import sys, subprocess, importlib.util

for pkg in ["xgboost", "shap"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("Environment ready.")

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    average_precision_score, roc_auc_score, precision_score,
    recall_score, f1_score, confusion_matrix,
    precision_recall_curve, roc_curve, classification_report
)
from xgboost import XGBClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

print("Libraries imported.")

## 1. Load the real fraud dataset

No synthetic replacement is used. The notebook downloads OpenML dataset **43627** when run.

In [ ]:
OPENML_DATA_ID = 43627
fraud_bunch = fetch_openml(data_id=OPENML_DATA_ID, as_frame=True)
df = fraud_bunch.frame.copy()

# Normalise target handling across OpenML representations.
if "Class" not in df.columns:
    if fraud_bunch.target is None:
        raise RuntimeError("Class target was not returned by OpenML.")
    df["Class"] = pd.Series(fraud_bunch.target, index=df.index)

df["Class"] = pd.to_numeric(df["Class"], errors="coerce")
df = df.dropna(subset=["Class"]).copy()
df["Class"] = df["Class"].astype(int)

print(f"Raw shape: {df.shape}")
print(f"Fraud cases: {int(df['Class'].sum()):,}")
print(f"Fraud rate: {df['Class'].mean():.4%}")
display(df.head())

In [ ]:
# Data-quality audit
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "unique": df.nunique(dropna=False)
})

print(f"Missing values: {int(df.isna().sum().sum()):,}")
print(f"Exact duplicate rows: {int(df.duplicated().sum()):,}")
display(quality)

# Remove exact duplicates before splitting to avoid identical records crossing sets.
df = df.drop_duplicates().reset_index(drop=True)
print(f"Shape after duplicate removal: {df.shape}")

## 2. Exploratory analysis — why accuracy is misleading

Fraud is a rare-event problem. A classifier that predicts every transaction as legitimate can still appear extremely accurate, so the project prioritises **precision-recall performance and operational capture**.

In [ ]:
class_counts = df["Class"].value_counts().sort_index()
labels = ["Legitimate", "Fraud"]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(labels, class_counts.values)
ax.set_title("Extreme class imbalance")
ax.set_ylabel("Transactions")
for i, v in enumerate(class_counts.values):
    ax.text(i, v, f"{v:,}\n({v/len(df):.3%})", ha="center", va="bottom")
plt.tight_layout()
plt.show()

naive_accuracy = 1 - df["Class"].mean()
print(f"Always predicting 'legitimate' would still achieve {naive_accuracy:.3%} accuracy.")

if "Amount" in df.columns:
    display(df.groupby("Class")["Amount"].agg(["count","mean","median","max"]))

    fig, ax = plt.subplots(figsize=(9, 4.5))
    for cls, label in [(0, "Legitimate"), (1, "Fraud")]:
        vals = np.log1p(df.loc[df["Class"] == cls, "Amount"].astype(float))
        ax.hist(vals, bins=60, alpha=0.55, density=True, label=label)
    ax.set_title("Transaction amount distribution")
    ax.set_xlabel("log(1 + Amount)")
    ax.set_ylabel("Density")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 3. Leakage-safe train / validation / test design

Model choice and threshold tuning happen on **validation data only**. The test set is used once at the end.

In [ ]:
train_df, temp_df = train_test_split(
    df, test_size=0.40, stratify=df["Class"], random_state=RANDOM_STATE
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["Class"], random_state=RANDOM_STATE
)

feature_cols = [c for c in df.columns if c != "Class"]

X_train, y_train = train_df[feature_cols].copy(), train_df["Class"].copy()
X_val, y_val = val_df[feature_cols].copy(), val_df["Class"].copy()
X_test, y_test = test_df[feature_cols].copy(), test_df["Class"].copy()

# Only scale magnitude/time fields; PCA variables are already transformed.
scale_cols = [c for c in ["Amount", "Time"] if c in feature_cols]
if scale_cols:
    scaler = RobustScaler()
    X_train.loc[:, scale_cols] = scaler.fit_transform(X_train[scale_cols])
    X_val.loc[:, scale_cols] = scaler.transform(X_val[scale_cols])
    X_test.loc[:, scale_cols] = scaler.transform(X_test[scale_cols])

split_summary = pd.DataFrame({
    "rows": [len(train_df), len(val_df), len(test_df)],
    "frauds": [int(y_train.sum()), int(y_val.sum()), int(y_test.sum())],
    "fraud_rate": [y_train.mean(), y_val.mean(), y_test.mean()]
}, index=["train", "validation", "test"])

display(split_summary)
print(f"Feature count: {len(feature_cols)}")

## 4. Train three complementary risk models

In [ ]:
# Model 1 — class-weighted Logistic Regression
logit = LogisticRegression(
    class_weight="balanced",
    max_iter=2500,
    random_state=RANDOM_STATE
)
logit.fit(X_train, y_train)
val_prob_logit = logit.predict_proba(X_val)[:, 1]

# Model 2 — XGBoost
neg, pos = np.bincount(y_train.astype(int))
scale_pos_weight = neg / pos

xgb = XGBClassifier(
    n_estimators=350,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=2,
    reg_lambda=2.0,
    reg_alpha=0.05,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
xgb.fit(X_train, y_train)
val_prob_xgb = xgb.predict_proba(X_val)[:, 1]

# Model 3 — anomaly signal trained only on legitimate behaviour
normal_train = X_train.loc[y_train == 0]
if len(normal_train) > 100_000:
    normal_train = normal_train.sample(100_000, random_state=RANDOM_STATE)

iso = IsolationForest(
    n_estimators=250,
    contamination="auto",
    random_state=RANDOM_STATE,
    n_jobs=-1
)
iso.fit(normal_train)

val_anomaly_raw = -iso.score_samples(X_val)
anom_lo, anom_hi = np.percentile(val_anomaly_raw, [1, 99])
val_anomaly = np.clip(
    (val_anomaly_raw - anom_lo) / (anom_hi - anom_lo + 1e-12), 0, 1
)

# Supervised probability stays dominant; anomaly score adds novelty sensitivity.
val_prob_hybrid = 0.85 * val_prob_xgb + 0.15 * val_anomaly

print(f"XGBoost scale_pos_weight: {scale_pos_weight:.1f}")
print("All models fitted.")

## 5. Validation model comparison

For extreme imbalance, **PR-AUC** is the primary selection metric.

In [ ]:
def evaluate_probabilities(name, y_true, prob, threshold=0.5):
    pred = (np.asarray(prob) >= threshold).astype(int)
    return {
        "model": name,
        "PR_AUC": average_precision_score(y_true, prob),
        "ROC_AUC": roc_auc_score(y_true, prob),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "F1": f1_score(y_true, pred, zero_division=0),
        "alerts": int(pred.sum())
    }

probability_map = {
    "Logistic Regression": val_prob_logit,
    "XGBoost": val_prob_xgb,
    "XGBoost + anomaly signal": val_prob_hybrid
}

val_results = pd.DataFrame([
    evaluate_probabilities(name, y_val, prob)
    for name, prob in probability_map.items()
]).set_index("model").sort_values("PR_AUC", ascending=False)

display(val_results.style.format({
    "PR_AUC":"{:.4f}", "ROC_AUC":"{:.4f}",
    "precision":"{:.4f}", "recall":"{:.4f}", "F1":"{:.4f}"
}))

best_model_name = val_results.index[0]
val_prob_best = probability_map[best_model_name]
print(f"Selected model on validation PR-AUC: {best_model_name}")

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.5))
for name, prob in probability_map.items():
    precision, recall, _ = precision_recall_curve(y_val, prob)
    ap = average_precision_score(y_val, prob)
    ax.plot(recall, precision, label=f"{name} (AP={ap:.3f})")
ax.set_title("Validation precision-recall curves")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8.5, 5.5))
for name, prob in probability_map.items():
    fpr, tpr, _ = roc_curve(y_val, prob)
    auc = roc_auc_score(y_val, prob)
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")
ax.plot([0,1],[0,1], linestyle="--", linewidth=1)
ax.set_title("Validation ROC curves")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

## 6. Business-cost threshold optimisation

A fixed 0.50 threshold is rarely appropriate for fraud. We choose an operating threshold using explicit illustrative costs:
- **£5** analyst cost per false alert
- missed fraud cost = **transaction amount + £100** fixed operational impact

In [ ]:
FALSE_ALERT_REVIEW_COST = 5.0
MISSED_FRAUD_FIXED_COST = 100.0

val_amount = (
    val_df["Amount"].astype(float).to_numpy()
    if "Amount" in val_df.columns
    else np.full(len(val_df), 100.0)
)

def decision_cost(y_true, prob, threshold, amounts):
    y = np.asarray(y_true).astype(int)
    pred = (np.asarray(prob) >= threshold).astype(int)
    fp = (pred == 1) & (y == 0)
    fn = (pred == 0) & (y == 1)
    return float(
        fp.sum() * FALSE_ALERT_REVIEW_COST
        + (amounts[fn] + MISSED_FRAUD_FIXED_COST).sum()
    )

thresholds = np.linspace(0.001, 0.999, 500)
costs = np.array([
    decision_cost(y_val, val_prob_best, t, val_amount)
    for t in thresholds
])
best_threshold = float(thresholds[np.argmin(costs)])

print(f"Cost-optimised validation threshold: {best_threshold:.3f}")
print(f"Minimum illustrative validation cost: £{costs.min():,.2f}")

fig, ax = plt.subplots(figsize=(9, 4.8))
ax.plot(thresholds, costs)
ax.axvline(best_threshold, linestyle="--", label=f"Best={best_threshold:.3f}")
ax.set_title("Validation decision cost vs threshold")
ax.set_xlabel("Decision threshold")
ax.set_ylabel("Illustrative cost (£)")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

cm = confusion_matrix(y_val, (val_prob_best >= best_threshold).astype(int))
fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(cm)
ax.set_title("Validation confusion matrix")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_xticks([0,1], ["Legit","Fraud"])
ax.set_yticks([0,1], ["Legit","Fraud"])
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm[i,j]:,}", ha="center", va="center")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## 7. Investigator-capacity analysis — fraud capture@K

Fraud teams have finite review capacity. This converts predictions into an operational question: **how much fraud is captured if analysts can only inspect the top K transactions?**

In [ ]:
def capture_at_k(y_true, score, k_values=(25,50,100,250,500,1000)):
    ranked = pd.DataFrame({
        "y": np.asarray(y_true),
        "score": np.asarray(score)
    }).sort_values("score", ascending=False).reset_index(drop=True)

    total_fraud = ranked["y"].sum()
    rows = []
    for k in k_values:
        k_eff = min(k, len(ranked))
        found = int(ranked.head(k_eff)["y"].sum())
        rows.append({
            "review_capacity": k_eff,
            "frauds_found": found,
            "precision_at_k": found / k_eff,
            "fraud_capture_rate": found / total_fraud if total_fraud else np.nan
        })
    return pd.DataFrame(rows)

capture_val = capture_at_k(y_val, val_prob_best)
display(capture_val.style.format({
    "precision_at_k":"{:.2%}",
    "fraud_capture_rate":"{:.2%}"
}))

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.plot(
    capture_val["review_capacity"],
    capture_val["fraud_capture_rate"] * 100,
    marker="o"
)
ax.set_title("Fraud capture vs investigator capacity")
ax.set_xlabel("Transactions manually reviewed")
ax.set_ylabel("Fraud captured (%)")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 8. Explainability with XGBoost importance + SHAP

In [ ]:
importance = pd.Series(
    xgb.feature_importances_,
    index=feature_cols
).sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(8, 6))
importance.sort_values().plot(kind="barh", ax=ax)
ax.set_title("Top XGBoost feature importances")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()
display(importance.to_frame("importance"))

import shap
sample_for_shap = X_val.sample(min(1500, len(X_val)), random_state=RANDOM_STATE)
explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(sample_for_shap)
shap.summary_plot(shap_values, sample_for_shap, max_display=15, show=False)
plt.title("SHAP summary — drivers of fraud risk")
plt.tight_layout()
plt.show()

## 9. One-time final test evaluation

The selected model and threshold are now frozen.

In [ ]:
test_prob_logit = logit.predict_proba(X_test)[:, 1]
test_prob_xgb = xgb.predict_proba(X_test)[:, 1]

test_anomaly_raw = -iso.score_samples(X_test)
test_anomaly = np.clip(
    (test_anomaly_raw - anom_lo) / (anom_hi - anom_lo + 1e-12), 0, 1
)
test_prob_hybrid = 0.85 * test_prob_xgb + 0.15 * test_anomaly

test_probability_map = {
    "Logistic Regression": test_prob_logit,
    "XGBoost": test_prob_xgb,
    "XGBoost + anomaly signal": test_prob_hybrid
}
test_prob_best = test_probability_map[best_model_name]

test_result = evaluate_probabilities(
    f"{best_model_name} @ frozen threshold",
    y_test, test_prob_best, best_threshold
)
display(pd.DataFrame([test_result]).set_index("model").style.format({
    "PR_AUC":"{:.4f}", "ROC_AUC":"{:.4f}",
    "precision":"{:.4f}", "recall":"{:.4f}", "F1":"{:.4f}"
}))

print(classification_report(
    y_test,
    (test_prob_best >= best_threshold).astype(int),
    target_names=["Legitimate","Fraud"],
    zero_division=0
))

test_amount = (
    test_df["Amount"].astype(float).to_numpy()
    if "Amount" in test_df.columns
    else np.full(len(test_df), 100.0)
)
test_cost = decision_cost(
    y_test, test_prob_best, best_threshold, test_amount
)
capture_test = capture_at_k(y_test, test_prob_best)

print(f"Illustrative test decision cost: £{test_cost:,.2f}")
display(capture_test.style.format({
    "precision_at_k":"{:.2%}",
    "fraud_capture_rate":"{:.2%}"
}))

## 10. Build the fraud-investigation queue

`actual_fraud` is shown only for retrospective evaluation. It would not be known when a live alert is created.

In [ ]:
queue = test_df.copy()
queue["risk_score"] = test_prob_best
queue["xgb_probability"] = test_prob_xgb
queue["anomaly_score"] = test_anomaly
queue["alert"] = (queue["risk_score"] >= best_threshold).astype(int)

pct_rank = queue["risk_score"].rank(method="first", pct=True)
queue["risk_tier"] = pd.cut(
    pct_rank,
    bins=[0, 0.98, 0.995, 0.999, 1.0],
    labels=["Low","Medium","High","Critical"],
    include_lowest=True
)

queue = queue.rename(columns={"Class":"actual_fraud"})
queue = queue.sort_values("risk_score", ascending=False)

show_cols = [
    c for c in [
        "Amount","risk_score","xgb_probability","anomaly_score",
        "risk_tier","alert","actual_fraud"
    ] if c in queue.columns
]

print("Top 25 transactions for manual investigation:")
display(queue[show_cols].head(25).style.format({
    "risk_score":"{:.4f}",
    "xgb_probability":"{:.4f}",
    "anomaly_score":"{:.4f}"
}))

queue[show_cols].to_csv("fraud_investigation_queue.csv", index=True)
capture_test.to_csv("test_capacity_analysis.csv", index=False)
val_results.to_csv("validation_model_comparison.csv")
print("Operational artefacts exported.")

## 11. Executive summary + production roadmap

In [ ]:
k100 = capture_test.loc[
    capture_test["review_capacity"] == min(100, len(test_df))
].iloc[0]

print(f"""
EXECUTIVE FRAUD-RISK SUMMARY
----------------------------
Selected model: {best_model_name}
Frozen threshold: {best_threshold:.3f}
Test PR-AUC: {test_result['PR_AUC']:.4f}
Test ROC-AUC: {test_result['ROC_AUC']:.4f}
Precision: {test_result['precision']:.2%}
Recall: {test_result['recall']:.2%}
Alerts generated: {test_result['alerts']:,}
Top-100 precision: {k100['precision_at_k']:.2%}
Top-100 fraud capture: {k100['fraud_capture_rate']:.2%}
Illustrative decision cost: £{test_cost:,.2f}

The project treats fraud detection as a decision system rather than
an accuracy competition: model selection is isolated from testing,
thresholds are tied to explicit costs, and predictions become a
ranked investigation queue for finite analyst capacity.
""")

### Production extensions
- streaming/event-time features;
- card-device-merchant-IP entity graphs;
- probability calibration and delayed-label handling;
- feature and fraud-rate drift monitoring;
- champion/challenger deployment;
- analyst feedback loops;
- customer-friction and fairness checks;
- alert-volume, latency and cost SLAs;
- model registry and reproducible feature pipelines.

### Interview defence
**“I treated fraud as a rare-event decision problem. I separated model selection from final testing, compared supervised and anomaly-based signals, optimised an explicit operating threshold, measured fraud capture at fixed investigation capacity, and used SHAP to explain model risk.”**